# Part 3: Metrics + Threshold — Where to Set the Operating Point
**⏱ This section takes approximately 30 minutes.**

---

## Scenario: Thursday — The Accuracy is a Lie

By Wednesday evening Sarah had a trained logistic regression with cross-validated accuracy of 88%. Today she's nervous, because:

> *"Accuracy 88% sounds great. But the baseline — guessing 'stayed' for everyone — is also 88%. So either the model is doing nothing useful, or accuracy is hiding the truth."*

She's about to discover the truth, and it'll change everything about what she presents to Marcus on Friday.

**By the end of this notebook you will be able to:**
- Read a confusion matrix and explain what each cell means in business terms
- Compute precision, recall, and F1 by hand and with sklearn
- See why accuracy is a misleading metric on imbalanced data
- Pick a classification threshold that matches a business constraint, not just the default 0.5

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, precision_recall_curve,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 4.5)

print("✅ Libraries loaded — including sklearn.metrics")

## Step 1 — Rebuild the trained pipeline

(Same code path as yesterday. We re-fit so this notebook is self-contained.)

In [ ]:
df = pd.read_csv("data/northstar_churn.csv")
y  = df["churned"]
X  = df.drop(columns=["customer_id", "churned"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42,
)

numeric_features = ["age", "tenure_months", "num_purchases_quarter",
                    "avg_monthly_spend_gbp", "returns_per_purchase",
                    "last_login_days_ago", "avg_review_polarity",
                    "support_tickets_quarter"]
categorical_features = ["region", "subscription_tier"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_features),
])

full_pipeline = Pipeline([
    ("prep",  preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
full_pipeline.fit(X_train, y_train)

print(f"Pipeline ready.  Test-set accuracy: {full_pipeline.score(X_test, y_test):.3f}")

## Step 2 — Predict probabilities, not just labels

`.predict()` returns 0/1 labels using the default 0.5 threshold. `.predict_proba()` returns the model's underlying probability of class 1 for every customer. We're going to work with probabilities so we can move the threshold.

In [ ]:
# Probabilities of class 1 (= churned)
y_proba = full_pipeline.predict_proba(X_test)[:, 1]
y_pred_default = (y_proba >= 0.5).astype(int)

print(f"y_proba shape: {y_proba.shape}")
print()
print("Sample predicted probabilities for the first 10 test customers:")
sample = pd.DataFrame({
    "customer_idx": X_test.index[:10],
    "actual":       y_test.iloc[:10].values,
    "prob(churn)":  y_proba[:10].round(3),
    "label@0.5":    y_pred_default[:10],
})
print(sample.to_string(index=False))

## Step 3 — The confusion matrix at threshold 0.5

Here is the moment of truth. Group every test-set prediction into one of four bins:

|  | Predicted **STAYED** | Predicted **CHURN** |
|---|---|---|
| **Actually STAYED** | TN — correctly let stayer alone | FP — wasted intervention |
| **Actually CHURN** | FN — missed real churner | TP — caught a real churner |

In [ ]:
cm = confusion_matrix(y_test, y_pred_default)
tn, fp, fn, tp = cm.ravel()

print(f"Confusion matrix @ threshold = 0.5:")
print(f"                    pred=stayed   pred=churn")
print(f"actual=stayed       TN = {tn:>4}    FP = {fp:>4}")
print(f"actual=churn        FN = {fn:>4}    TP = {tp:>4}")
print()

# Visualise
fig, ax = plt.subplots(figsize=(6, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=["stayed", "churn"])
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion matrix @ threshold = 0.5")
plt.tight_layout()
plt.show()

# Compute the four headline metrics
acc  = accuracy_score(y_test, y_pred_default)
prec = precision_score(y_test, y_pred_default, zero_division=0)
rec  = recall_score(y_test, y_pred_default, zero_division=0)
f1   = f1_score(y_test, y_pred_default, zero_division=0)

print(f"Metrics @ 0.5:")
print(f"  Accuracy:   {acc:.3f}")
print(f"  Precision:  {prec:.3f}   (of all flagged, fraction actually churners)")
print(f"  Recall:     {rec:.3f}   (of all actual churners, fraction we flagged)")
print(f"  F1:         {f1:.3f}")

### 💡 What you should notice

This is the heart of the lesson:

- **Accuracy ≈ 88%** — looks great on paper.
- **But recall is shockingly low** — the model only catches a small fraction of actual churners. The vast majority of churners slip through as false negatives.
- **Precision is OK** — when the model DOES say "churn", it's usually right. But it rarely says "churn" at all.

**What's happening:** at threshold 0.5, the model is being *conservative*. It only predicts "churn" when its probability output exceeds 0.5 — which is hard for a logistic regression on imbalanced data. It's playing it safe by mostly predicting "stayed".

**Sarah's reaction:** "*If I show Marcus 88% accuracy, he'll be pleased. Then a churner will leave, he'll ask why we didn't flag them, and I'll have nothing to say. I need to lower the threshold.*"

## ⏸️ Pause and Predict

Before we sweep across thresholds, predict:

- If we lower the threshold from 0.5 to 0.3 (flag anyone with > 30% predicted churn probability), what happens to:
  - **Recall** — will it go up, down, or stay the same?
  - **Precision** — same question?
  - **Accuracy** — same question?
- What's the *minimum* recall Sarah should accept before walking into Marcus's office?

*Write your predictions here:*

> *Sample answers:*
> - **Recall ↑** — we'll flag more customers, including more actual churners. Recall climbs.
> - **Precision ↓** — but a bigger fraction of the additional flags will be false positives. Precision drops.
> - **Accuracy ↓** — when the model says "churn" more, it's wrong more often on stayers. Accuracy slips because the data is so imbalanced.
> - **Minimum acceptable recall** — context-dependent, but anything below 50% means we're missing more than we catch. For an "early warning" system, 70% is a reasonable target.

## Step 4 — Sweep across thresholds

Let's see precision, recall, and F1 as functions of the threshold. The right operating point depends on which trade-off Sarah is willing to make.

In [ ]:
thresholds = np.arange(0.05, 0.95, 0.02)
records = []

for t in thresholds:
    pred = (y_proba >= t).astype(int)
    if pred.sum() == 0:
        records.append((t, np.nan, 0.0, np.nan, pred.sum()))
        continue
    records.append((
        t,
        precision_score(y_test, pred, zero_division=0),
        recall_score(y_test, pred, zero_division=0),
        f1_score(y_test, pred, zero_division=0),
        pred.sum(),
    ))

sweep = pd.DataFrame(records, columns=["threshold", "precision", "recall", "F1", "n_flagged"])

# Plot
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(sweep["threshold"], sweep["precision"], label="Precision", linewidth=2, color="steelblue")
ax.plot(sweep["threshold"], sweep["recall"],    label="Recall",    linewidth=2, color="coral")
ax.plot(sweep["threshold"], sweep["F1"],        label="F1",        linewidth=2, color="seagreen", linestyle="--")
ax.axvline(0.5, color="gray", linestyle=":", linewidth=1, label="Default threshold (0.5)")
ax.axvline(0.3, color="indianred", linestyle=":", linewidth=1.5, label="Sarah's recommendation (0.3)")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision / Recall / F1 as the threshold changes")
ax.legend(loc="lower left")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print("Threshold      Precision    Recall      F1     N flagged")
for t in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
    row = sweep.iloc[(sweep["threshold"] - t).abs().idxmin()]
    print(f"  {row['threshold']:.2f}         {row['precision']:.3f}      {row['recall']:.3f}     {row['F1']:.3f}     {int(row['n_flagged'])}")

### 💡 What this tells us

- **Threshold 0.5 (default)** — flags very few customers. Precision is decent but recall is poor.
- **Threshold 0.3 (Sarah's choice)** — flags many more customers. Recall climbs steeply. Precision drops but is still reasonable.
- **F1 peaks somewhere between 0.2 and 0.4** — that's the threshold that *balances* precision and recall.
- **The "best" threshold depends on the business question.** If retention team capacity is the constraint, pick the threshold that gives you the right *number of flagged customers* — not the one that maximises any single metric.

## Step 5 — Make the business-aware choice

NorthStar's retention team can call about 200 customers a week from the test set's worth of data (≈ 10% of test). Sarah needs the threshold that surfaces ~200 customers and maximises recall within that capacity.

In [ ]:
# Pick the threshold so the number flagged is approximately the team's capacity
target_capacity = 200  # in the test set
best_threshold = sweep.iloc[(sweep["n_flagged"] - target_capacity).abs().idxmin()]["threshold"]

# Apply
y_pred_business = (y_proba >= best_threshold).astype(int)

cm_b = confusion_matrix(y_test, y_pred_business)
tn_b, fp_b, fn_b, tp_b = cm_b.ravel()

print(f"Chosen threshold: {best_threshold:.2f}")
print(f"Number flagged:   {int(y_pred_business.sum())}  (team capacity ~ {target_capacity})")
print()
print(f"Confusion matrix @ threshold {best_threshold:.2f}:")
print(f"                    pred=stayed   pred=churn")
print(f"actual=stayed       TN = {tn_b:>4}    FP = {fp_b:>4}")
print(f"actual=churn        FN = {fn_b:>4}    TP = {tp_b:>4}")
print()

# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, t, p in zip(axes, [0.5, best_threshold], [y_pred_default, y_pred_business]):
    cm_x = confusion_matrix(y_test, p)
    ConfusionMatrixDisplay(cm_x, display_labels=["stayed", "churn"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d",
    )
    ax.set_title(f"Threshold = {t:.2f}")

plt.tight_layout()
plt.show()

# Compare metrics
print(f"\nDecision summary:")
print(f"{'':22s}  threshold=0.5    threshold={best_threshold:.2f}")
for metric_fn, name in [(accuracy_score,  "Accuracy"),
                         (precision_score, "Precision"),
                         (recall_score,    "Recall"),
                         (f1_score,        "F1")]:
    kwargs = {"zero_division": 0} if name != "Accuracy" else {}
    v1 = metric_fn(y_test, y_pred_default,  **kwargs)
    v2 = metric_fn(y_test, y_pred_business, **kwargs)
    arrow = "↑" if v2 > v1 else ("↓" if v2 < v1 else "→")
    print(f"  {name:20s}      {v1:.3f}             {v2:.3f}  {arrow}")

### 💡 What this tells us — and what Sarah will say to Marcus

- **At threshold 0.5** — the model surfaces only a handful of customers. It "looks accurate" (88%) but is operationally useless: it catches barely any real churners.
- **At threshold ~0.3** — Sarah surfaces ~200 customers (matching retention team's capacity), catches a much higher fraction of the real churners (recall climbs), at the cost of more false positives (precision drops).
- **Accuracy went DOWN** in absolute terms when we lowered the threshold — but accuracy was the wrong metric all along.

**Back to our scenario:**
> Sarah's recommendation to Marcus: *"Use the model at threshold 0.3. We flag about 200 customers a week. We catch the majority of the real churners. We accept that some of those flagged customers wouldn't have left anyway — that's the cost of getting recall up."*

**What Marcus will ask next:** *"How much money does this actually save?"* That cost-benefit analysis lives in `optional_extensions.ipynb`.

## ✅ Section Summary

| What we learned | Key takeaway |
|---|---|
| **Accuracy is misleading** on imbalanced datasets — `88%` was just `1 - churn_rate` | Always pair accuracy with precision/recall |
| **Confusion matrix** tells you exactly where the model gets things wrong (FP vs FN) | Read it before trusting any single metric |
| **Precision vs recall** is a tradeoff — choose based on business cost of each error | No "right answer" without context |
| **F1** balances precision and recall — useful when both errors hurt | Reach for this when in doubt |
| **Threshold choice** is a business decision, not a technical one | Capacity + cost dictate where to set it |

---

## 🏁 Friday — What Sarah Presents to Marcus

| Item | Number |
|---|---|
| **Model** | One sklearn `Pipeline` — preprocess + logistic regression. Same code path at training and prediction. |
| **Cross-validated accuracy** | 88% (but irrelevant — see below) |
| **Recommended operating threshold** | **0.30** — matches retention team capacity (~200 customers/week) |
| **At threshold 0.30** | Recall higher than at 0.5; precision lower but still actionable |
| **Top predictive features** | High returns, low review polarity, short tenure, non-premium tier |

Marcus listens. He nods. Then he asks:

> *"Sarah — this is the first model we own. Can you make it BETTER next week? We're hitting capacity. If you could squeeze more recall without more false positives, we'd save real money. Try those tree-based models you mentioned."*

That question — **"can you make it better?"** — is the engine of **L04 (Advanced Supervised Learning).**

---

**Next step →** Open `assignment.ipynb` to apply this pipeline to a new domain (Lakeside Bank).

*Or, for deeper material on bias-variance math, ROC-AUC, learning curves, and manual feature engineering: open `optional_extensions.ipynb`.*

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## Extension 1 — The precision-recall curve

The sklearn `precision_recall_curve` function returns the precision and recall at every possible threshold. Plot precision vs recall to see the *frontier* of possible operating points.

In [ ]:
prec_curve, rec_curve, thr_curve = precision_recall_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(rec_curve, prec_curve, linewidth=2, color="steelblue")
ax.fill_between(rec_curve, prec_curve, alpha=0.1, color="steelblue")
ax.axhline(y_test.mean(), color="gray", linestyle="--", linewidth=1,
           label=f"Random baseline (= class rate = {y_test.mean():.2f})")

# Mark the two thresholds
for t, label, color in [(0.5, "T=0.5", "navy"), (0.3, "T=0.3", "indianred")]:
    pred = (y_proba >= t).astype(int)
    p_t  = precision_score(y_test, pred, zero_division=0)
    r_t  = recall_score(y_test, pred, zero_division=0)
    ax.scatter([r_t], [p_t], color=color, s=120, zorder=5, edgecolor="white", linewidth=2)
    ax.annotate(label, (r_t, p_t), xytext=(10, 10), textcoords="offset points",
                fontweight="bold", fontsize=12, color=color)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision–Recall Curve (test set)")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

print("The PR curve traces the achievable (precision, recall) pairs as the threshold varies.")
print("Operating closer to the top-right = better model. Below the dashed line = worse than random.")

## Extension 2 — F1 vs threshold — finding the maximiser

If you don't have an explicit capacity constraint, F1 is the natural metric to maximise. Plot F1 against threshold and pick the peak.

In [ ]:
best_idx = sweep["F1"].idxmax()
best_row = sweep.iloc[best_idx]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(sweep["threshold"], sweep["F1"], linewidth=2, color="seagreen")
ax.axvline(best_row["threshold"], color="indianred", linestyle="--",
           label=f"Max F1 = {best_row['F1']:.3f} at threshold = {best_row['threshold']:.2f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("F1 score")
ax.set_title("F1 score across thresholds — the peak is the 'balanced' choice")
ax.legend()
ax.set_ylim(0, max(0.5, best_row["F1"] * 1.2))
plt.tight_layout()
plt.show()

print(f"F1-optimal threshold: {best_row['threshold']:.2f}")
print(f"At that threshold:    precision = {best_row['precision']:.3f}, recall = {best_row['recall']:.3f}, F1 = {best_row['F1']:.3f}")

## Extension 3 — Cost-sensitive thresholding (a thought experiment)

If you can put a £ value on each false positive (FP) and false negative (FN), the optimal threshold is the one that *minimises expected cost*. Quick example:

- FP cost: £20 (one wasted retention call)
- FN cost: £150 (a lost customer's lifetime value)

Then the cost function for any threshold is `total_cost = 20·FP + 150·FN`.

In [ ]:
COST_FP = 20.0    # £ per wasted call
COST_FN = 150.0   # £ per lost customer

costs = []
for t in np.arange(0.05, 0.95, 0.01):
    pred = (y_proba >= t).astype(int)
    cm_t = confusion_matrix(y_test, pred)
    _, fp, fn, _ = cm_t.ravel()
    cost = COST_FP * fp + COST_FN * fn
    costs.append((t, fp, fn, cost))

cost_df = pd.DataFrame(costs, columns=["threshold", "FP", "FN", "expected_cost_gbp"])
best_cost_row = cost_df.iloc[cost_df["expected_cost_gbp"].idxmin()]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(cost_df["threshold"], cost_df["expected_cost_gbp"], linewidth=2, color="indianred")
ax.axvline(best_cost_row["threshold"], color="navy", linestyle="--",
           label=f"Min cost = £{best_cost_row['expected_cost_gbp']:.0f} at T = {best_cost_row['threshold']:.2f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Expected total cost (£)")
ax.set_title(f"Cost-optimal threshold (FP cost = £{COST_FP:.0f} · FN cost = £{COST_FN:.0f})")
ax.legend()
plt.tight_layout()
plt.show()

print(f"With FN cost much higher than FP cost, the optimal threshold drops below 0.5")
print(f"to catch more real churners — even at the cost of more wasted calls.")
print()
print(f"At T = {best_cost_row['threshold']:.2f}: FP = {int(best_cost_row['FP'])}, FN = {int(best_cost_row['FN'])}, total cost = £{best_cost_row['expected_cost_gbp']:.0f}")